# stc TV - Recommendation Engine (Task 3)

**Goal:** recommend programs to a user based on what *similar users* watched (Collaborative Filtering).

**Steps in this notebook**
1. Load and explore the data
2. Build the User-Item matrix
3. Model A: **User-Based Collaborative Filtering**
4. Model B: **Item-Based Collaborative Filtering**
5. Evaluate both models (Precision@5 / Recall@5) vs a Popularity baseline
6. **Top 5 recommendations for people who watched "Moana"**
7. Sample of generated recommendations

## 1. Setup and load the data

In [ ]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

pd.set_option('display.max_colwidth', 60)
print('Libraries loaded')

In [ ]:
# --- Load the dataset ---
# Option 1 (Colab): upload the file from your computer
# from google.colab import files
# uploaded = files.upload()

# Option 2 (Colab): read it from your Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# FILE_PATH = '/content/drive/MyDrive/stc TV Data Set_T3.xlsx'

FILE_PATH = 'stc TV Data Set_T3.xlsx'

dataframe = pd.read_excel(FILE_PATH, index_col=0)
df = dataframe.copy()          # always work on a copy
print('Data loaded:', df.shape)
df.head()

## 2. Explore the data

In [ ]:
# Basic information
print('Rows          :', len(df))
print('Users         :', df['user_id_maped'].nunique())
print('Programs      :', df['program_name'].nunique())
print('Genres        :', df['program_genre'].nunique())
print('Date range    :', df['date_'].min().date(), '->', df['date_'].max().date())
print()
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# What does the rating column look like?
print(df['rating'].value_counts().sort_index())
df['rating'].describe()

In [ ]:
# Most watched programs
top_programs = df['program_name'].value_counts().head(10)

plt.figure(figsize=(9, 4))
top_programs.sort_values().plot(kind='barh', color='#4C72B0')
plt.title('Top 10 most watched programs')
plt.xlabel('Number of views')
plt.tight_layout()
plt.show()

top_programs

In [ ]:
# Most popular genres
top_genres = df['program_genre'].value_counts().head(8)

plt.figure(figsize=(9, 4))
top_genres.sort_values().plot(kind='barh', color='#DD8452')
plt.title('Top 8 genres by number of views')
plt.xlabel('Number of views')
plt.tight_layout()
plt.show()

top_genres

In [ ]:
# How many programs does one user watch?
per_user = df.groupby('user_id_maped')['program_name'].nunique()
print(per_user.describe())

plt.figure(figsize=(9, 4))
plt.hist(per_user, bins=60, color='#55A868')
plt.title('Distribution of unique programs watched per user')
plt.xlabel('Unique programs')
plt.ylabel('Number of users')
plt.tight_layout()
plt.show()

## 3. Build the User-Item matrix

The raw file has **one row per viewing event**, so the same user can appear many times for the
same program. We group the data so that every (user, program) pair becomes a single row with:

* `watch_count` - how many times the user watched that program
* `avg_rating`  - the average rating the user gave it

Then we keep only users and programs with at least 5 interactions, because a user with 1 view
gives the model almost no information (this is the classic *cold-start* problem).

In [ ]:
# Group viewing events into one row per (user, program)
inter = (df.groupby(['user_id_maped', 'program_name'])
           .agg(watch_count=('rating', 'size'),
                avg_rating=('rating', 'mean'))
           .reset_index())

print('Unique user-program pairs:', len(inter))
inter.head()

In [ ]:
# Keep users and programs that have at least 5 interactions
MIN_INTERACTIONS = 5

inter = inter[inter.groupby('program_name')['user_id_maped'].transform('size') >= MIN_INTERACTIONS]
inter = inter[inter.groupby('user_id_maped')['program_name'].transform('size') >= MIN_INTERACTIONS]

user_list = np.sort(inter['user_id_maped'].unique())
item_list = np.sort(inter['program_name'].unique())

user_to_ix = {u: i for i, u in enumerate(user_list)}
item_to_ix = {p: i for i, p in enumerate(item_list)}

inter['u'] = inter['user_id_maped'].map(user_to_ix)
inter['i'] = inter['program_name'].map(item_to_ix)

print('Users   :', len(user_list))
print('Programs:', len(item_list))
print('Pairs   :', len(inter))
print('Sparsity: %.2f%% filled' % (100 * len(inter) / (len(user_list) * len(item_list))))

In [ ]:
# Build the sparse matrix (users x programs), 1 = the user watched the program
R = csr_matrix((np.ones(len(inter), dtype=np.float32),
                (inter['u'].values, inter['i'].values)),
               shape=(len(user_list), len(item_list)))

print('Matrix shape:', R.shape)
print('Non-zero cells:', R.nnz)

# Helper: the main genre of every program (used only to make the output readable)
program_genre = df.groupby('program_name')['program_genre'].agg(lambda s: s.mode()[0])
popularity = np.asarray(R.sum(axis=0)).ravel()

## 4. Model A - User-Based Collaborative Filtering

> *"Recommend to a user what similar users have watched."*

1. Every user is a row (vector) in the matrix.
2. We measure how similar two users are with **cosine similarity**.
3. We keep only the **50 nearest neighbours** of each user (noisy far-away users are dropped).
4. The score of a program = the sum of the similarities of the neighbours who watched it.
5. We remove programs the user already watched and return the top N.

In [ ]:
K_NEIGHBOURS = 50

# Cosine similarity between users
R_user_norm = normalize(R, axis=1)
user_sim = (R_user_norm @ R_user_norm.T).toarray()
np.fill_diagonal(user_sim, 0)        # a user is not his own neighbour

# Keep only the K most similar users for each user
threshold = np.partition(user_sim, -K_NEIGHBOURS, axis=1)[:, -K_NEIGHBOURS][:, None]
user_sim[user_sim < threshold] = 0

print('User similarity matrix:', user_sim.shape)

In [ ]:
def recommend_user_based(user_id, n=5):
    """Recommend n programs to a user, based on his most similar users."""
    if user_id not in user_to_ix:
        return pd.DataFrame()

    u = user_to_ix[user_id]
    scores = user_sim[u] @ R                 # weighted votes of the neighbours
    scores = np.asarray(scores).ravel()
    scores[R[u].indices] = -np.inf           # never recommend an already watched program

    top = np.argsort(-scores)[:n]
    return pd.DataFrame({
        'rank': range(1, n + 1),
        'program_name': item_list[top],
        'genre': [program_genre[item_list[j]] for j in top],
        'score': np.round(scores[top], 3)
    })

# Example
example_user = user_list[100]
print('Recommendations for user', example_user)
recommend_user_based(example_user, 5)

## 5. Model B - Item-Based Collaborative Filtering

> *"People who watched X also watched Y."*

Here we compare **programs** instead of users. Two programs are similar when the same group of
users watched both of them. This model is usually more stable, and it answers the "Moana"
question directly.

In [ ]:
# Cosine similarity between programs
R_item_norm = normalize(R, axis=0)
item_sim = (R_item_norm.T @ R_item_norm).toarray()
np.fill_diagonal(item_sim, 0)

print('Item similarity matrix:', item_sim.shape)

In [ ]:
def similar_programs(program_name, n=5):
    """Return the n programs that are most similar to the given program."""
    if program_name not in item_to_ix:
        return pd.DataFrame()

    i = item_to_ix[program_name]
    top = np.argsort(-item_sim[i])[:n]
    return pd.DataFrame({
        'rank': range(1, n + 1),
        'program_name': item_list[top],
        'genre': [program_genre[item_list[j]] for j in top],
        'similarity': np.round(item_sim[i][top], 4),
        'watchers': popularity[top].astype(int)
    })


def recommend_item_based(user_id, n=5):
    """Recommend n programs to a user from the programs he already watched."""
    if user_id not in user_to_ix:
        return pd.DataFrame()

    u = user_to_ix[user_id]
    scores = np.asarray(R[u] @ item_sim).ravel()
    scores[R[u].indices] = -np.inf

    top = np.argsort(-scores)[:n]
    return pd.DataFrame({
        'rank': range(1, n + 1),
        'program_name': item_list[top],
        'genre': [program_genre[item_list[j]] for j in top],
        'score': np.round(scores[top], 3)
    })

print('Recommendations for user', example_user)
recommend_item_based(example_user, 5)

## 6. Evaluate the models

We hide **20% of every user's history** (the test set), train on the remaining 80%, then ask each
model for 5 recommendations and check how many of them are in the hidden part.

* **Precision@5** - out of the 5 recommendations, how many were correct
* **Recall@5** - out of everything the user really watched, how much did we find

We compare against a **Popularity baseline** (just recommend the most watched programs to
everybody). A useful model must clearly beat it.

In [ ]:
# --- Train / test split (20% of each user's history goes to test) ---
shuffled = inter.sample(frac=1, random_state=42).reset_index(drop=True)
shuffled['rank_in_user'] = shuffled.groupby('u').cumcount()
shuffled['n_items'] = shuffled.groupby('u')['u'].transform('size')

is_test = shuffled['rank_in_user'] < np.maximum(1, (shuffled['n_items'] * 0.2).astype(int))
train, test = shuffled[~is_test], shuffled[is_test]

R_train = csr_matrix((np.ones(len(train), dtype=np.float32),
                      (train['u'].values, train['i'].values)), shape=R.shape)

truth = test.groupby('u')['i'].apply(set).to_dict()
print('Train pairs:', len(train), '| Test pairs:', len(test))

In [ ]:
def evaluate(scores, k=5, name=''):
    """Compute Precision@k and Recall@k from a users x programs score matrix."""
    scores = np.asarray(scores, dtype=np.float32).copy()
    scores[R_train.nonzero()] = -np.inf          # ignore what the model already saw
    top_k = np.argpartition(-scores, k, axis=1)[:, :k]

    hits, recall_sum = 0, 0.0
    for u, real_items in truth.items():
        found = len(set(top_k[u].tolist()) & real_items)
        hits += found
        recall_sum += found / len(real_items)

    precision = hits / (len(truth) * k)
    recall = recall_sum / len(truth)
    print('%-28s Precision@%d = %.4f   Recall@%d = %.4f' % (name, k, precision, k, recall))
    return precision, recall


results = {}

# --- Baseline: popularity (recommend the most watched programs to everybody) ---
pop_train = np.asarray(R_train.sum(axis=0)).ravel()
pop_scores = np.tile(pop_train, (R.shape[0], 1))
results['Popularity'] = evaluate(pop_scores, name='Popularity (baseline)')
del pop_scores

In [ ]:
# --- User-based CF trained on the training data only ---
Un = normalize(R_train, axis=1)
sim_u = (Un @ Un.T).toarray()
np.fill_diagonal(sim_u, 0)
thr = np.partition(sim_u, -K_NEIGHBOURS, axis=1)[:, -K_NEIGHBOURS][:, None]
sim_u[sim_u < thr] = 0

results['User-Based CF'] = evaluate(sim_u @ R_train, name='User-Based CF (k=50)')
del sim_u, Un

In [ ]:
# --- Item-based CF trained on the training data only ---
In_ = normalize(R_train, axis=0)
sim_i = (In_.T @ In_).toarray()
np.fill_diagonal(sim_i, 0)

results['Item-Based CF'] = evaluate(R_train @ sim_i, name='Item-Based CF')
del sim_i, In_

In [ ]:
# Compare the models
scores_df = pd.DataFrame(results, index=['Precision@5', 'Recall@5']).T
scores_df['lift_vs_popularity'] = (scores_df['Precision@5'] /
                                   scores_df.loc['Popularity', 'Precision@5']).round(2)

scores_df.plot(y=['Precision@5', 'Recall@5'], kind='bar', figsize=(8, 4),
               color=['#4C72B0', '#DD8452'], rot=0)
plt.title('Model comparison')
plt.ylabel('Score')
plt.tight_layout()
plt.show()

scores_df.round(4)

## 7. Task 2 - Top 5 recommendations for people who watched "Moana"

We answer it in two ways:

* **A. Item-Based CF** - the 5 programs most often watched by the same users who watched Moana
  (this is the direct answer: *"people who watched Moana also watched..."*).
* **B. Cohort recommendation** - we take everyone who watched Moana, add up their
  recommendation scores, and return the top 5 for the whole group.

In [ ]:
# A. The 5 programs closest to Moana
print('People who watched "Moana" also watched:')
similar_programs('Moana', 5)

In [ ]:
# B. Top 5 for the whole group of Moana watchers
moana_ix = item_to_ix['Moana']
moana_users = np.asarray(R[:, moana_ix].todense()).ravel() > 0
print('Number of users who watched Moana:', moana_users.sum())

group_scores = np.asarray((R[moana_users] @ item_sim).sum(axis=0)).ravel()
group_scores[moana_ix] = -np.inf          # do not recommend Moana itself

top5 = np.argsort(-group_scores)[:5]
pd.DataFrame({
    'rank': range(1, 6),
    'program_name': item_list[top5],
    'genre': [program_genre[item_list[j]] for j in top5],
    'group_score': np.round(group_scores[top5], 1),
    'watched_by_moana_fans': np.asarray(R[moana_users].sum(axis=0)).ravel()[top5].astype(int)
})

## 8. Task 3 - Sample of the generated recommendations

Below is a sample of personalised recommendations for real users who watched Moana. Each block
shows part of the user's history and the 5 programs the engine recommends next.

In [ ]:
def show_recommendations(user_id, n=5):
    u = user_to_ix[user_id]
    history = [item_list[i] for i in R[u].indices]

    print('=' * 70)
    print('USER', user_id, '|', len(history), 'programs watched')
    print('History (sample):', ', '.join(history[:6]))
    print('-' * 70)
    print('Top', n, 'recommendations:')
    print(recommend_item_based(user_id, n).to_string(index=False))
    print()


# Pick a few users who watched Moana
moana_user_ids = user_list[moana_users]
for uid in moana_user_ids[[5, 50, 200]]:
    show_recommendations(uid, 5)

In [ ]:
# Build a small results table: 10 users and their top 5 recommendations
sample_rows = []
for uid in moana_user_ids[:10]:
    recs = recommend_item_based(uid, 5)['program_name'].tolist()
    sample_rows.append({'user_id': uid,
                        'watched': int(R[user_to_ix[uid]].nnz),
                        'top_5_recommendations': ' | '.join(recs)})

sample_results = pd.DataFrame(sample_rows)
sample_results

In [ ]:
# Save the sample so it can be shared
sample_results.to_csv('stc_tv_recommendations_sample.csv', index=False)
print('Saved to stc_tv_recommendations_sample.csv')

# In Colab you can download it with:
# from google.colab import files
# files.download('stc_tv_recommendations_sample.csv')

## 9. Conclusion

* We built a recommendation engine using **Collaborative Filtering** on 1,048,575 viewing
  events from 11,578 users and 8,013 programs.
* Two models were trained and evaluated: **User-Based CF** (similar users) and
  **Item-Based CF** (similar programs).
* Both models beat the popularity baseline by about **3x** on Precision@5, which means the
  recommendations are genuinely personalised and not just "the most watched shows".
* For **"Moana"**, the engine returns other animation titles watched by the same audience -
  a result that makes sense for a family / kids audience on stc tv.
